# 01. 데이터 수집

**프로젝트**: 창원시 폭우 침수·하수 역류 우선 대응지역 분석  
**목적**: 분석에 필요한 공공데이터 다운로드 및 기상청 API 수집

---

## 1. 환경 설정

In [ ]:
import os
import sys
import pandas as pd
import requests
from pathlib import Path
from dotenv import load_dotenv

# 프로젝트 루트를 path에 추가
PROJECT_ROOT = Path.cwd().parent
sys.path.insert(0, str(PROJECT_ROOT))

# .env 파일에서 API 키 로드
load_dotenv(PROJECT_ROOT / '.env')

# 경로 설정
RAW_DIR = PROJECT_ROOT / 'data' / 'raw'
EXTERNAL_DIR = PROJECT_ROOT / 'data' / 'external'

print(f'프로젝트 루트: {PROJECT_ROOT}')
print(f'원본 데이터 저장: {RAW_DIR}')

## 2. 공공데이터 다운로드 가이드

아래 데이터를 수동으로 다운로드하여 `data/raw/` 폴더에 저장합니다.

| # | 데이터 | 출처 | URL | 파일명 |
|---|--------|------|-----|--------|
| 1 | 하수관로 시설별 설치현황 | 한국환경공단 | [data.go.kr](https://www.data.go.kr/data/15118453/fileData.do) | `sewer_pipe_status.csv` |
| 2 | 시간별 강수량 | 기상청 | [data.go.kr](https://www.data.go.kr/data/15150315/fileData.do) | `hourly_rainfall.csv` |
| 3 | 배수펌프장 현황 | 환경부 | [data.go.kr](https://www.data.go.kr/data/15047868/fileData.do) | `drainage_pump.csv` |
| 4 | 하수도 배수구역 | 국토교통부 | [data.go.kr](https://www.data.go.kr/data/15129161/fileData.do) | `drainage_area.csv` |
| 5 | 건축물대장 (창원시) | 창원시 | [data.go.kr](https://www.data.go.kr/data/15064338/fileData.do) | `building_register.csv` |
| 6 | 동별 인구통계 | 통계청 | [kosis.kr](https://kosis.kr) | `population_by_dong.csv` |
| 7 | 하수도 민원 상세 | 창원시 하수도사업소 | 정보공개청구 | `sewer_complaints.csv` |

> ⚠️ **7번 하수도 민원 데이터**는 정보공개청구 후 수령 (약 10영업일)

## 3. 데이터 다운로드 확인

In [ ]:
# 필요한 파일 목록
REQUIRED_FILES = {
    'sewer_pipe_status.csv': '하수관로 시설별 설치현황',
    'hourly_rainfall.csv': '시간별 강수량',
    'drainage_pump.csv': '배수펌프장 현황',
    'drainage_area.csv': '하수도 배수구역',
    'building_register.csv': '건축물대장',
    'population_by_dong.csv': '동별 인구통계',
}

OPTIONAL_FILES = {
    'sewer_complaints.csv': '하수도 민원 상세 (정보공개청구)',
}

print('=== 필수 데이터 확인 ===')
for fname, desc in REQUIRED_FILES.items():
    path = RAW_DIR / fname
    status = '✅ 확인' if path.exists() else '❌ 미확보'
    print(f'  {status}  {desc} ({fname})')

print('\n=== 선택 데이터 확인 ===')
for fname, desc in OPTIONAL_FILES.items():
    path = RAW_DIR / fname
    status = '✅ 확인' if path.exists() else '⏳ 대기'
    print(f'  {status}  {desc} ({fname})')

## 4. 기상청 API 데이터 수집

기상청 단기예보 및 관측 데이터를 API로 수집합니다.  
API 키는 `.env` 파일의 `DATA_GO_KR_API_KEY`에 설정합니다.

In [ ]:
API_KEY = os.getenv('DATA_GO_KR_API_KEY')

if not API_KEY:
    print('⚠️ .env 파일에 DATA_GO_KR_API_KEY를 설정해주세요.')
    print('   공공데이터포털(data.go.kr) 가입 후 API 키 발급')
else:
    print(f'✅ API 키 확인: {API_KEY[:8]}...')

In [ ]:
def fetch_weather_observation(api_key, start_date, end_date, station_id='155'):
    """
    기상청 종관기상관측(ASOS) 데이터 수집
    station_id 155 = 창원 관측소
    """
    # TODO: 기상청 API 엔드포인트 확인 후 구현
    url = 'http://apis.data.go.kr/1360000/AsosHourlyInfoService/getWthrDataList'
    params = {
        'serviceKey': api_key,
        'numOfRows': '999',
        'pageNo': '1',
        'dataType': 'JSON',
        'dataCd': 'ASOS',
        'dateCd': 'HR',
        'startDt': start_date,
        'endDt': end_date,
        'startHh': '00',
        'endHh': '23',
        'stnIds': station_id,
    }
    
    response = requests.get(url, params=params)
    
    if response.status_code == 200:
        data = response.json()
        items = data.get('response', {}).get('body', {}).get('items', {}).get('item', [])
        return pd.DataFrame(items)
    else:
        print(f'API 오류: {response.status_code}')
        return pd.DataFrame()

# TODO: 실제 수집 실행
# df_weather = fetch_weather_observation(API_KEY, '20230101', '20241231')
# df_weather.to_csv(EXTERNAL_DIR / 'weather_observation.csv', index=False)
print('기상 데이터 수집 함수 준비 완료 (API 키 설정 후 실행)')

## 5. 기상청 특보 데이터 수집

In [ ]:
def fetch_weather_warnings(api_key, start_date, end_date):
    """
    기상청 특보(호우, 폭염 등) 데이터 수집
    """
    # TODO: 기상특보 API 구현
    url = 'http://apis.data.go.kr/1360000/WthrWrnInfoService/getWthrWrnList'
    params = {
        'serviceKey': api_key,
        'numOfRows': '100',
        'pageNo': '1',
        'dataType': 'JSON',
        'stnId': '155',  # 창원
        'fromTmFc': start_date,
        'toTmFc': end_date,
    }
    
    response = requests.get(url, params=params)
    
    if response.status_code == 200:
        data = response.json()
        items = data.get('response', {}).get('body', {}).get('items', {}).get('item', [])
        return pd.DataFrame(items)
    else:
        print(f'API 오류: {response.status_code}')
        return pd.DataFrame()

# TODO: 실제 수집 실행
# df_warnings = fetch_weather_warnings(API_KEY, '20230101', '20241231')
# df_warnings.to_csv(EXTERNAL_DIR / 'weather_warnings.csv', index=False)
print('기상 특보 수집 함수 준비 완료')

## 6. 수집 결과 요약

In [ ]:
# 수집된 데이터 파일 목록
print('=== data/raw/ ===')
for f in sorted(RAW_DIR.glob('*')):
    if f.name != 'README.md':
        size = f.stat().st_size / 1024
        print(f'  {f.name:40s} {size:>8.1f} KB')

print('\n=== data/external/ ===')
for f in sorted(EXTERNAL_DIR.glob('*')):
    if f.name != 'README.md':
        size = f.stat().st_size / 1024
        print(f'  {f.name:40s} {size:>8.1f} KB')